# Оценка согласованности разметки IMDb

Цель работы - оценить согласованность меток тональности, присвоенных отзывам
двумя аннотаторами

Исходные данные:
- `annotations/labeled_imdb_by_iskander.json`;
- `annotations/labeled_imdb_by_andrey.json`.

Сопоставление выполнено по идентификатору `data.item_id` с проверкой совпадения
текстов. До расчёта проверяются уникальность ID, полнота наборов и наличие одной
завершённой метки у каждого аннотатора. Допустимые метки по codebook:
`positive`, `negative`, `review_required`.

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.metrics import cohen_kappa_score

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "annotations/labeled_imdb_by_iskander.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Откройте ноутбук внутри репозитория imdb-cs2-data-augmentation.")

LABELS = ["positive", "negative", "review_required"]


def read_labels(path):
    """Прочитать по одной завершённой метке на каждый уникальный ID отзыва."""
    tasks = json.loads(path.read_text(encoding="utf-8-sig"))
    if not isinstance(tasks, list) or not tasks:
        raise ValueError(f"{path.name}: ожидался непустой список заданий Label Studio.")
    records = {}
    for task in tasks:
        data = task.get("data", {})
        item_id, review = data.get("item_id"), data.get("review")
        if not isinstance(item_id, str) or not item_id.strip():
            raise ValueError(f"{path.name}: отсутствует item_id.")
        if item_id in records:
            raise ValueError(f"{path.name}: повторяется ID {item_id}.")
        if not isinstance(review, str) or not review.strip():
            raise ValueError(f"{path.name}, {item_id}: пустой текст отзыва.")
        annotations = [a for a in task.get("annotations", [])
                       if not a.get("was_cancelled", False)]
        if len(annotations) != 1:
            raise ValueError(f"{path.name}, {item_id}: нужна одна завершённая аннотация.")
        results = [r for r in annotations[0].get("result", [])
                   if r.get("type") == "choices" and r.get("from_name") == "sentiment"]
        choices = results[0].get("value", {}).get("choices", []) if len(results) == 1 else []
        if len(choices) != 1 or choices[0] not in LABELS:
            raise ValueError(f"{path.name}, {item_id}: нужна одна допустимая метка sentiment.")
        records[item_id] = {"label": choices[0], "review": review}
    return records


def compare_labels(iskander, andrey):
    """Сопоставить одинаковые отзывы; неполные наборы нельзя сравнивать молча."""
    if iskander.keys() != andrey.keys():
        raise ValueError(
            f"Нет у Андрея: {sorted(iskander.keys() - andrey.keys())}; "
            f"нет у Искандера: {sorted(andrey.keys() - iskander.keys())}"
        )
    rows = []
    for item_id in sorted(iskander):
        a, b = iskander[item_id], andrey[item_id]
        if a["review"].split() != b["review"].split():
            raise ValueError(f"{item_id}: тексты отзывов отличаются между экспортами.")
        rows.append({"item_id": item_id, "iskander": a["label"], "andrey": b["label"],
                     "agree": a["label"] == b["label"], "review": a["review"]})
    return pd.DataFrame(rows)


iskander = read_labels(ROOT / "annotations/labeled_imdb_by_iskander.json")
andrey = read_labels(ROOT / "annotations/labeled_imdb_by_andrey.json")
comparison = compare_labels(iskander, andrey)
print(f"Проверены ID, тексты и метки {len(comparison)} отзывов у каждого участника.")

Проверены ID, тексты и метки 50 отзывов у каждого участника.


## Метрики

Рассчитаны доля совпадающих меток и коэффициент каппа Коэна:

$$\kappa = \frac{p_o-p_e}{1-p_e},$$

где $p_o$ - наблюдаемая доля совпадений, $p_e$ - ожидаемая доля случайных
совпадений, рассчитанная по распределениям меток двух аннотаторов.
При $p_e=1$ коэффициент не определён.

In [2]:
total = len(comparison)
matched = int(comparison["agree"].sum())
disagreements = comparison.loc[~comparison["agree"],
                               ["item_id", "iskander", "andrey", "review"]].copy()
only_one_label = len(set(comparison["iskander"]) | set(comparison["andrey"])) == 1
kappa = None if only_one_label else cohen_kappa_score(comparison["iskander"], comparison["andrey"])

print(f"Всего отзывов: {total}")
print(f"Совпали: {matched}; различаются: {total - matched}")
print(f"Процент совпадений: {matched / total:.2%}")
print("Каппа Коэна: не определена (оба выбрали только одну и ту же метку)."
      if kappa is None else f"Каппа Коэна: {kappa:.4f}")

print("\nСколько раз каждый участник выбрал каждую метку:")
display(pd.DataFrame({
    "Искандер": comparison["iskander"].value_counts(),
    "Андрей": comparison["andrey"].value_counts(),
}).reindex(LABELS, fill_value=0).fillna(0).astype(int))

print("Таблица совпадений: строки - Искандер, столбцы - Андрей.")
display(pd.crosstab(comparison["iskander"], comparison["andrey"])
        .reindex(index=LABELS, columns=LABELS, fill_value=0))

Всего отзывов: 50
Совпали: 50; различаются: 0
Процент совпадений: 100.00%
Каппа Коэна: 1.0000

Сколько раз каждый участник выбрал каждую метку:


,Искандер,Андрей
positive,25,25
negative,25,25
review_required,0,0


Таблица совпадений: строки - Искандер, столбцы - Андрей.


andrey,positive,negative,review_required
iskander,,,
positive,25,0,0
negative,0,25,0
review_required,0,0,0


## Расхождения

Выделены отзывы с различающимися метками. Исходные экспорты при сравнении не изменяются.

In [3]:
if disagreements.empty:
    print("Расхождений не обнаружено.")
else:
    with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
        display(disagreements)

Расхождений не обнаружено.


## Вывод

In [4]:
print(f"Сопоставлены метки {total} отзывов IMDb двух аннотаторов.")
print(f"Число совпадений: {matched}; число расхождений: {total - matched}.")
print(f"Доля совпадений: {matched / total:.2%}.")
print("Коэффициент каппа Коэна не определён."
      if kappa is None else f"Коэффициент каппа Коэна: {kappa:.4f}.")
print("На данной выборке получено полное совпадение меток."
      if disagreements.empty else "Выявленные расхождения требуют совместного разбора по codebook.")
print("Результаты характеризуют согласованность разметок, а не правильность меток.")


Сопоставлены метки 50 отзывов IMDb двух аннотаторов.
Число совпадений: 50; число расхождений: 0.
Доля совпадений: 100.00%.
Коэффициент каппа Коэна: 1.0000.
На данной выборке получено полное совпадение меток.
Результаты характеризуют согласованность разметок, а не правильность меток.
